# Day 3 — Python for DFIR & Automation

*Python for Security — 3-Day Intensive  |  Istidama Consulting*

**DELTA Stage:** Test + Advance

### Day Overview
Day 3 turns the week's skills into a forensics workflow: file metadata, timelines, and structured artifact data feed into a capstone triage script. The day ends with a documentation workshop and a short demo, so every learner leaves with a portfolio-ready artifact and a clean repo.

Each day in this notebook is organized in two parts: **Techniques** (worked, runnable examples you read and run) followed by **Practice — Your Turn** (the floor/ceiling exercises you complete yourself, gathered at the end).

---


## 3.1 Recap & Share-outs  *(15 min)*

**Objective:** Reconnect with Day 2 work before extending it into forensics.

**Steps:**

1. 2–3 volunteers give a 60-second demo of their Day 2 tool (in-person or remote).


## 3.2 Techniques Demonstrated: File Metadata, Timelines & Structured Artifacts  *(40 min — read along and run every cell)*

**Objective:** See each core technique work end to end before applying it yourself in Practice.

**What's demonstrated below, in order:**

1. Reading size and modified-time metadata with `os.stat()`.
2. Hashing files in a folder with `hashlib`.
3. Building and sorting a timeline from that metadata.
4. Flagging files modified within a recent time window.
5. Filtering CSV rows to a specific field value.
6. Extracting the hour from a timestamp and tallying with `Counter`.
7. Writing a summary back out with `csv.writer`.

These are the exact techniques you'll apply yourself — against different data — in the Practice section at the end of this notebook.


In [1]:
import os, time  # os for filesystem ops, time for timestamps

# Demo setup: a small folder with files "modified" at different times
os.makedirs("demo_evidence", exist_ok=True)  # create the demo folder, no error if it already exists
demo_files = [("alpha.txt", 7200), ("beta.bin", 120), ("gamma.docx", 172800)]  # (filename, age-in-seconds) pairs
now = time.time()  # capture the current time as a Unix timestamp
for name, age_seconds in demo_files:  # loop over each planned file and its desired age
    path = os.path.join("demo_evidence", name)  # build the full path for this file
    with open(path, "w") as f:  # create the file
        f.write(f"demo content for {name}\n")  # write placeholder content
    mtime = now - age_seconds  # compute the timestamp that is age_seconds in the past
    os.utime(path, (mtime, mtime))  # backdate the file's access and modified times

print("Demo evidence folder ready:", os.listdir("demo_evidence"))  # list the files just created


Demo evidence folder ready: ['alpha.txt', 'beta.bin', 'gamma.docx']


In [2]:
# Technique 1: os.stat() for size and modified time
for name in os.listdir("demo_evidence"):  # loop over every filename in the folder
    path = os.path.join("demo_evidence", name)  # build the full path for this file
    stat_result = os.stat(path)  # get filesystem metadata for the file
    print(name, "| size:", stat_result.st_size, "bytes", "| modified:", time.ctime(stat_result.st_mtime))  # print name, size, and modified time


alpha.txt | size: 28 bytes | modified: Fri Sep 11 12:52:20 2026
beta.bin | size: 27 bytes | modified: Fri Sep 11 14:50:20 2026
gamma.docx | size: 29 bytes | modified: Wed Sep  9 14:52:20 2026


In [3]:
import hashlib  # standard library module providing cryptographic hash functions

# Technique 2: hash every file in a folder
def hash_file(path):  # define a helper that hashes a single file's contents
    with open(path, "rb") as f:  # open the file in binary mode for hashing
        return hashlib.sha256(f.read()).hexdigest()  # compute and return its SHA-256 hash as a hex string

for name in os.listdir("demo_evidence"):  # loop over every filename in the folder
    print(name, "->", hash_file(os.path.join("demo_evidence", name))[:12], "...")  # print the first 12 hex chars of each hash


alpha.txt -> 8a5b7ffda821 ...
beta.bin -> cfd5ddaec04f ...
gamma.docx -> 211474184cd9 ...


In [4]:
# Technique 3: build a timeline (name, hash, mtime) and sort it
timeline = []  # accumulator list of (name, short_hash, mtime) tuples
for name in os.listdir("demo_evidence"):  # loop over every filename in the folder
    path = os.path.join("demo_evidence", name)  # build the full path for this file
    mtime = os.stat(path).st_mtime  # get the file's last-modified timestamp
    timeline.append((name, hash_file(path)[:12], mtime))  # record name, short hash, and mtime

timeline.sort(key=lambda entry: entry[2])  # sort the entries chronologically by mtime

for name, short_hash, mtime in timeline:  # loop over the sorted timeline
    print(f"{time.ctime(mtime)}  {name}  {short_hash}...")  # print a human-readable timeline row


Wed Sep  9 14:52:20 2026  gamma.docx  211474184cd9...
Fri Sep 11 12:52:20 2026  alpha.txt  8a5b7ffda821...
Fri Sep 11 14:50:20 2026  beta.bin  cfd5ddaec04f...


In [5]:
# Technique 4: flag files modified within a recent window
window_seconds = 3600  # only flag files modified within the last hour
for name, short_hash, mtime in timeline:  # loop over the timeline built above
    if now - mtime <= window_seconds:  # true if this file's age is within the window
        print("RECENT:", name, f"({int(now - mtime)}s ago)")  # report the recently modified file and its age


RECENT: beta.bin (120s ago)


In [6]:
import csv  # standard library module for reading/writing CSV files
from collections import Counter  # Counter tallies hashable items and supports most_common()

# Demo setup: a small artifact export, separate from the one you'll practice on later
demo_artifacts_csv = """timestamp,event_type,path
2026-02-10T08:15:00,file_created,/tmp/one.txt
2026-02-10T08:20:00,process_started,/usr/bin/bash
2026-02-10T09:05:00,file_created,/tmp/two.txt
2026-02-10T09:40:00,file_deleted,/tmp/one.txt
"""  # multi-line CSV string: header row plus four sample DFIR artifact events

with open("demo_artifacts.csv", "w") as f:  # open (create/overwrite) the demo artifacts CSV for writing
    f.write(demo_artifacts_csv)  # write the sample CSV text to disk

with open("demo_artifacts.csv") as f:  # reopen the CSV for reading
    rows = list(csv.DictReader(f))  # parse all rows into a list of dicts

# Technique 5: filter rows to a specific event type
created_only = [r for r in rows if r["event_type"] == "file_created"]  # keep only file_created events
print("file_created rows:", created_only)  # print the filtered rows


file_created rows: [{'timestamp': '2026-02-10T08:15:00', 'event_type': 'file_created', 'path': '/tmp/one.txt'}, {'timestamp': '2026-02-10T09:05:00', 'event_type': 'file_created', 'path': '/tmp/two.txt'}]


In [7]:
# Technique 6: extract the hour from a timestamp and tally with Counter
by_type = Counter(r["event_type"] for r in rows)  # tally how many rows have each event_type
by_hour = Counter(r["timestamp"][11:13] for r in rows)  # tally rows by the HH slice of their timestamp

print("by type:", by_type)  # print the per-event-type tally
print("by hour:", by_hour)  # print the per-hour tally


by type: Counter({'file_created': 2, 'process_started': 1, 'file_deleted': 1})
by hour: Counter({'08': 2, '09': 2})


In [8]:
# Technique 7: write a summary back out to a new CSV
with open("demo_summary.csv", "w", newline="") as f:  # open (create/overwrite) the summary CSV for writing
    writer = csv.writer(f)  # wrap the file in a CSV writer
    writer.writerow(["event_type", "count"])  # write the header row
    for event_type, count in by_type.items():  # loop over each event type and its tally
        writer.writerow([event_type, count])  # write one data row per event type

with open("demo_summary.csv") as f:  # reopen the summary CSV for reading
    print(f.read())  # print its full contents to confirm the write worked


event_type,count
file_created,2
process_started,1
file_deleted,1



**A note on `sys.argv`, used in the Practice 3 ceiling below:** it can't be demonstrated live inside a notebook, because a notebook kernel has no command line to read arguments from. The pattern you'll adapt into a real `.py` file looks like this:

```python
import sys, os

if __name__ == "__main__":
    folder = sys.argv[1] if len(sys.argv) > 1 else "sample_evidence"
    if not os.path.isdir(folder):
        print(f"Error: '{folder}' is not a folder.")
    else:
        run_triage(folder)
```

Save this as the bottom of a script (e.g. `triage.py`) and run it from your terminal as `python triage.py sample_evidence`.


## Examples

Work through the practice items below using the techniques demonstrated above. Floor/ceiling markers tell you the minimum bar and the stretch goal.


### 3.3 Practice: File Forensics Fundamentals  *(50 min)*

**Objective:** Extract file metadata and assemble a simple timeline.

**Steps:**

1. Run the setup cell below to generate a small sample folder with files of different ages.
2. Use os.stat() on each file to pull size and created/modified timestamps.
3. Hash each file with hashlib.
4. Sort files by modified time to build a simple timeline; print filename, hash, last-modified time.

> **Floor:** timeline printed correctly for the provided sample folder.
> **Ceiling:** timeline flags any file modified within a specific suspicious time window.


In [9]:
import os, time  # os for filesystem ops, time for timestamps

# Setup: a small sample folder with files "modified" at different times
os.makedirs("sample_evidence", exist_ok=True)  # create the sample folder, no error if it already exists
files_and_ages = [("notes.txt", 3600), ("payload.bin", 60), ("report.docx", 86400)]  # (filename, age-in-seconds) pairs
now = time.time()  # capture the current time as a Unix timestamp
for name, age_seconds in files_and_ages:  # loop over each planned file and its desired age
    path = os.path.join("sample_evidence", name)  # build the full path for this file
    with open(path, "w") as f:  # create the file
        f.write(f"sample content for {name}\n")  # write placeholder content
    mtime = now - age_seconds  # compute the timestamp that is age_seconds in the past
    os.utime(path, (mtime, mtime))  # backdate the file's access and modified times
print("Sample evidence folder ready:", os.listdir("sample_evidence"))  # list the files just created


Sample evidence folder ready: ['notes.txt', 'payload.bin', 'report.docx']


In [10]:
import hashlib  # standard library module providing cryptographic hash functions

# Floor: build a timeline -- filename, hash, last-modified time, sorted by modified time
def hash_file(path):  # define a helper that hashes a single file's contents
    with open(path, "rb") as f:  # open the file in binary mode for hashing
        return hashlib.sha256(f.read()).hexdigest()  # compute and return its SHA-256 hash as a hex string

timeline = []  # accumulator list of (name, short_hash, mtime) tuples
for name in os.listdir("sample_evidence"):  # loop over every filename in the folder
    path = os.path.join("sample_evidence", name)  # build the full path for this file
    mtime = os.stat(path).st_mtime  # get the file's last-modified timestamp
    timeline.append((name, hash_file(path)[:12], mtime))  # record name, short hash, and mtime

timeline.sort(key=lambda entry: entry[2])  # sort the entries chronologically by mtime

for name, short_hash, mtime in timeline:  # loop over the sorted timeline
    print(f"{time.ctime(mtime)}  {name}  {short_hash}...")  # print a human-readable timeline row


Thu Sep 10 14:53:06 2026  report.docx  9b061022b2f3...
Fri Sep 11 13:53:06 2026  notes.txt  13d0f715fc93...
Fri Sep 11 14:52:06 2026  payload.bin  78649806a835...


In [11]:
# Ceiling: flag files modified within the last 10 minutes (600 seconds)
suspicious_window_seconds = 600  # only flag files modified within the last 10 minutes
for name, short_hash, mtime in timeline:  # loop over the timeline built above
    if now - mtime <= suspicious_window_seconds:  # true if this file's age is within the window
        print("RECENT:", name, f"({int(now - mtime)}s ago)")  # report the recently modified file and its age


RECENT: payload.bin (60s ago)


### 3.4 Practice: Structured Forensic Data Lab  *(50 min)*

**Objective:** Work with a semi-structured artifact export.

**Materials:** sample_artifacts.csv (generated below)

**Steps:**

1. Run the setup cell to generate sample_artifacts.csv.
2. Filter rows to a specific event type (e.g., “file_created”).
3. Summarize counts by event type and by hour.

> **Floor:** filtered and summarized output produced correctly.
> **Ceiling:** summary is exported to a new, clean CSV or report file.


In [12]:
# Setup: generate a sample artifact export
artifacts_csv = """timestamp,event_type,path
2026-01-04T09:00:01,process_started,/usr/bin/bash
2026-01-04T09:01:15,file_created,/tmp/dropper.sh
2026-01-04T09:01:20,file_created,/tmp/payload.bin
2026-01-04T09:05:44,process_started,/tmp/dropper.sh
2026-01-04T10:02:10,file_created,/home/user/report.docx
2026-01-04T10:03:00,file_deleted,/tmp/payload.bin
"""  # multi-line CSV string: header row plus six sample DFIR artifact events

with open("sample_artifacts.csv", "w") as f:  # open (create/overwrite) the artifacts CSV for writing
    f.write(artifacts_csv)  # write the sample CSV text to disk


In [13]:
import csv  # standard library module for reading/writing CSV files
from collections import Counter  # Counter tallies hashable items and supports most_common()

# Floor: filter to file_created events and summarize by event type and by hour
with open("sample_artifacts.csv") as f:  # open the artifacts CSV for reading
    rows = list(csv.DictReader(f))  # parse all rows into a list of dicts

created_only = [r for r in rows if r["event_type"] == "file_created"]  # keep only file_created events
print("file_created rows:", created_only)  # print the filtered rows

by_type = Counter(r["event_type"] for r in rows)  # tally how many rows have each event_type
by_hour = Counter(r["timestamp"][11:13] for r in rows)  # tally rows by the HH slice of their timestamp

print("by type:", by_type)  # print the per-event-type tally
print("by hour:", by_hour)  # print the per-hour tally


file_created rows: [{'timestamp': '2026-01-04T09:01:15', 'event_type': 'file_created', 'path': '/tmp/dropper.sh'}, {'timestamp': '2026-01-04T09:01:20', 'event_type': 'file_created', 'path': '/tmp/payload.bin'}, {'timestamp': '2026-01-04T10:02:10', 'event_type': 'file_created', 'path': '/home/user/report.docx'}]
by type: Counter({'file_created': 3, 'process_started': 2, 'file_deleted': 1})
by hour: Counter({'09': 4, '10': 2})


In [14]:
# Ceiling: export the summary to a new CSV
with open("summary_report.csv", "w", newline="") as f:  # open (create/overwrite) the report CSV for writing
    writer = csv.writer(f)  # wrap the file in a CSV writer
    writer.writerow(["event_type", "count"])  # write the header row
    for event_type, count in by_type.items():  # loop over each event type and its tally
        writer.writerow([event_type, count])  # write one data row per event type

with open("summary_report.csv") as f:  # reopen the report CSV for reading
    print(f.read())  # print its full contents to confirm the write worked


event_type,count
process_started,2
file_created,3
file_deleted,1



### 3.5 Practice: Capstone Build — End-to-End Triage Script  *(90 min)*

**Objective:** Combine the week's skills into one working triage tool.

**Steps:**

1. Plan first: list what the script needs to collect — running processes, recent file changes, file hashes.
2. Build incrementally: process check → recent-file-changes check → hash check → combine into one report function.
3. Test the combined script against the provided sample environment.

> **Floor:** script runs top to bottom and prints a combined report without crashing.
> **Ceiling:** script accepts a command-line argument (e.g., a target folder) and handles a missing-file error gracefully.


In [15]:
def check_processes():  # define a function that gathers a sample of running processes
    """Return a short list of running process names/pids."""
    try:
        import psutil  # optional third-party library for cross-platform process listing
        return [p.info for p in psutil.process_iter(["pid", "name"])][:5]  # grab pid/name for the first 5 processes
    except ImportError:  # psutil isn't installed -- fall back to a shell command
        import subprocess  # standard library module for running external commands
        result = subprocess.run(["ps", "aux"], capture_output=True, text=True)  # run `ps aux` and capture its output
        return result.stdout.splitlines()[:5]  # return the first 5 lines of output

def check_recent_files(folder, window_seconds=600):  # define a function finding recently modified files
    """Return files in `folder` modified within the last `window_seconds`."""
    current_time = time.time()  # capture the current time as a Unix timestamp
    recent = []  # accumulator list of (name, age_seconds) tuples
    for name in os.listdir(folder):  # loop over every filename in the folder
        path = os.path.join(folder, name)  # build the full path for this file
        mtime = os.stat(path).st_mtime  # get the file's last-modified timestamp
        if current_time - mtime <= window_seconds:  # true if this file's age is within the window
            recent.append((name, int(current_time - mtime)))  # record the filename and its age in seconds
    return recent  # hand back the list of recently modified files

def check_hashes(folder):  # define a function hashing every file in a folder
    """Return {filename: sha256} for every file in `folder`."""
    hashes = {}  # accumulator dict of filename -> hash
    for name in os.listdir(folder):  # loop over every filename in the folder
        path = os.path.join(folder, name)  # build the full path for this file
        with open(path, "rb") as f:  # open the file in binary mode for hashing
            hashes[name] = hashlib.sha256(f.read()).hexdigest()  # compute and store its SHA-256 hash
    return hashes  # hand back the filename -> hash mapping

def run_triage(folder="sample_evidence"):  # define the function combining all checks into one report
    """Combine the checks above into one printed report. (floor)"""
    print(f"--- Triage report for '{folder}' ---")  # print a report header naming the target folder

    print("\nRunning processes (sample):")  # section header for the process listing
    for p in check_processes():  # loop over the sampled processes
        print(" ", p)  # print each process entry

    print("\nRecently modified files:")  # section header for recent-file activity
    recent = check_recent_files(folder)  # gather files modified within the default window
    if recent:  # only enter this branch if at least one recent file was found
        for name, age in recent:  # loop over each recent file and its age
            print(f"  {name} ({age}s ago)")  # print the file and how recently it changed
    else:  # no recently modified files were found
        print("  none")  # report that explicitly

    print("\nFile hashes:")  # section header for the hash listing
    for name, digest in check_hashes(folder).items():  # loop over every file and its computed hash
        print(f"  {name}: {digest[:12]}...")  # print the filename and the first 12 hex chars of its hash

run_triage()  # run the full triage report against the default sample_evidence folder


--- Triage report for 'sample_evidence' ---

Running processes (sample):
  {'pid': 0, 'name': 'System Idle Process'}
  {'pid': 4, 'name': 'System'}
  {'pid': 224, 'name': 'Registry'}
  {'pid': 760, 'name': 'smss.exe'}
  {'pid': 920, 'name': 'csrss.exe'}

Recently modified files:
  payload.bin (96s ago)

File hashes:
  notes.txt: 13d0f715fc93...
  payload.bin: 78649806a835...
  report.docx: 9b061022b2f3...


In [16]:
import sys  # standard library module providing access to command-line arguments

# Ceiling: accept a target folder as a command-line argument and handle a missing folder gracefully
if __name__ == "__main__":  # only run this block when the script is executed directly
    folder = sys.argv[1] if len(sys.argv) > 1 else "sample_evidence"  # use the first CLI arg, or a default
    if not os.path.isdir(folder):  # the given path doesn't exist or isn't a directory
        print(f"Error: '{folder}' is not a folder.")  # report the problem instead of crashing
    else:  # the folder is valid
        run_triage(folder)  # run the triage report against it


Error: '--f=c:\Users\fadi\AppData\Roaming\jupyter\runtime\kernel-v3e06492c45b0b389f90e3b87986684e91c79b61da.json' is not a folder.


## 3.6 Documentation Workshop  *(60 min)*

**Objective:** Write a README another analyst could follow without the author in the room.

**Steps:**

1. Compare a “bad README” vs. “good README” example (below).
2. Draft your own: purpose, requirements, how to run, example output, known limitations.
3. Peer swap: can your partner run your script using only your README?

> **Floor:** README covers purpose and how to run the script.
> **Ceiling:** README also includes example output and known limitations.


**Bad README example:**
```markdown
run the script and it does the thing
```

**Good README example:**
```markdown
## Triage Script

**Purpose:** Collects running processes, recently modified files, and file hashes into one report.

**Requirements:** Python 3.11+, no external packages.

**How to run:**
\`\`\`bash
python triage.py sample_evidence
\`\`\`

**Example output:**
\`\`\`
Running processes: 42 found
Recently modified files: payload.bin (3 min ago)
\`\`\`

**Known limitations:** process listing requires psutil; falls back to `ps aux` on systems without it.
```

Draft your own README in the cell below (as a markdown cell or a `README.md` file in your repo).

_Your README draft goes here._

## 3.7 Final Commit, Push, Demo  *(45 min)*

**Objective:** Close out the repo and show finished work.

**Steps:**

1. Final commit and push of the triage script and README.
2. 90-second lightning demo per learner or pair, alternating in-person and remote presenters.


```bash
git add .
git commit -m "Add end-to-end triage script and README"
git push
```


## 3.8 Cohort Debrief & Hand-off  *(30 min)*

**Objective:** Close the intensive and connect it to what comes next.

**Steps:**

1. Walk through the grading rubric against the three deliverables produced this week.
2. Preview how this baseline feeds into the Phase 2 core curriculum.
3. Complete the exit survey.


## 3.9 Exercises — On Your Own

New problems, separate from the labs and capstone above. Use the techniques from section 3.2 (`os.stat()`, `hashlib`, timelines, `csv`, `Counter`). No worked example to copy from this time.

> Do these as homework, or as a fast-finisher extension during 3.5.


**Exercise 1 — Largest file.** Using `sample_evidence/` (already on disk from 3.3), find and print the name and size in bytes of the *largest* file. *Hint: `os.stat(path).st_size`; track the biggest as you loop, or use `max()` with a `key=`.*

In [18]:
# Exercise 1: find the largest file in sample_evidence/ by size
import os

target_dir = "sample_evidence"

largest_file = None
max_size = -1

for filename in os.listdir(target_dir):
    filepath = os.path.join(target_dir, filename)
    
    # التأكد من أنه ملف وليس مجلدًا فرعيًا
    if os.path.isfile(filepath):
        file_size = os.stat(filepath).st_size
        if file_size > max_size:
            max_size = file_size
            largest_file = filename

print(f"Largest file: {largest_file}")
print(f"Size: {max_size} bytes")


Largest file: payload.bin
Size: 32 bytes


**Exercise 2 — Busiest path.** Using `exercise_artifacts.csv` (generated below), find which `path` value appears most often across all events, and print it along with its count. *Hint: this is the same `Counter` pattern as tallying `event_type`, just applied to a different column.*

In [20]:
# Setup: a fresh artifact export for this exercise
exercise_artifacts_csv = """timestamp,event_type,path
2026-03-02T14:00:00,process_started,/usr/bin/bash
2026-03-02T14:01:10,file_created,/tmp/loader.sh
2026-03-02T14:01:40,process_started,/tmp/loader.sh
2026-03-02T14:02:05,file_created,/tmp/loader.sh
2026-03-02T14:03:20,process_started,/tmp/loader.sh
2026-03-02T14:05:00,file_deleted,/tmp/loader.sh
"""  # multi-line CSV string: header row plus six events, mostly repeating the same path

with open("exercise_artifacts.csv", "w") as f:  # open (create/overwrite) the exercise CSV for writing
    f.write(exercise_artifacts_csv)  # write the sample CSV text to disk

import csv  # standard library module for reading/writing CSV files
from collections import Counter  # Counter tallies hashable items and supports most_common()

# Exercise 2: find the path value that appears most often
# قراءة ملف CSV وتجميع المسارات
with open("exercise_artifacts.csv", "r") as f:
    reader = csv.DictReader(f)
    paths = [row["path"] for row in reader]

# حساب المسار الأكثر تكراراً
path_counts = Counter(paths)
most_common_path, count = path_counts.most_common(1)[0]

print(f"Busiest path: {most_common_path}")
print(f"Count: {count}")


Busiest path: /tmp/loader.sh
Count: 5


**Exercise 3 — Stretch: duplicate-content detector.** In real triage, two differently-named files with the *same* hash are worth flagging — it can mean a payload was copied or renamed to evade detection. Write `find_duplicates(folder)` that hashes every file in `folder`, groups filenames by hash, and returns only the groups with more than one file (a `dict` of `hash -> [filenames]`). Run it against `dup_evidence/` (generated below) and print any duplicate groups found.

In [22]:
# Setup: a folder with one duplicated payload under two different names
os.makedirs("dup_evidence", exist_ok=True) # create the folder, no error if it already exists
with open(os.path.join("dup_evidence", "invoice.docx"), "w") as f: # create the first duplicate file
    f.write("identical payload content\n") # write content identical to svchost.exe below
with open(os.path.join("dup_evidence", "svchost.exe"), "w") as f: # create the second duplicate file
    f.write("identical payload content\n") # write content identical to invoice.docx above
with open(os.path.join("dup_evidence", "notes.txt"), "w") as f: # create an unrelated third file
    f.write("unrelated content\n") # write different content so this file won't match the duplicates

import hashlib # standard library module providing cryptographic hash functions
from collections import defaultdict

def find_duplicates(folder):
    """Return {hash: [filenames]} for every hash shared by more than one file."""
    hashes = defaultdict(list)
    
    for filename in os.listdir(folder):
        filepath = os.path.join(folder, filename)
        if os.path.isfile(filepath):
            with open(filepath, "rb") as f:
                file_hash = hashlib.sha256(f.read()).hexdigest()
                hashes[file_hash].append(filename)
                
    return {h: files for h, files in hashes.items() if len(files) > 1}

# Call and print results
results = find_duplicates("dup_evidence")
print(results)


{'135f61e09e66e4be50b44674ef96a523aec3af6ce173df491d57a698eb6b3250': ['invoice.docx', 'svchost.exe']}
